# Exercícios — Agentes Colaborativos (Notebook 9)

Este notebook de exercícios reforça os conceitos do `notebooks/encontro9.ipynb`, focando em colaboração de agentes com LangGraph e uso de LLMs via LangChain.

Objetivos:
- Construir nós de agentes com `ChatPromptTemplate` e `StrOutputParser`.
- Orquestrar fluxos com `StateGraph` e executar exemplos.
- Ajustar prompts para obter respostas úteis e consistentes.

Observação: evite funções determinísticas — os agentes devem usar o LLM.


## Pré-requisitos
- Configure `GOOGLE_API_KEY` no ambiente ou `.env`.
- Certifique-se de instalar dependências (veja `requirements.txt` do projeto).
- O catálogo para RAG está em `ingestion/products_catalog.md` (usado em alguns exercícios).


In [ ]:
import os
from typing import TypedDict

from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# Utilitários do projeto (se precisar RAG)
try:
    from projetos.commercial_bot.rag import build_rag  # quando executado a partir da raiz
except Exception:
    try:
        from rag import build_rag  # quando executado dentro do módulo
    except Exception:
        build_rag = None

def build_llm():
    load_dotenv()
    api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")
    if not api_key:
        raise ValueError("Defina GOOGLE_API_KEY/GEMINI_API_KEY no ambiente ou .env")
    model_name = os.environ.get("MODEL_NAME", "gemini-2.0-flash")
    return ChatGoogleGenerativeAI(model=model_name, temperature=0, google_api_key=api_key)

llm = build_llm()
parser = StrOutputParser()


## Exercício 1 — Pesquisador & Escritor (Cooperativo)
Implemente dois agentes: um pesquisador que cria um resumo objetivo e um escritor que gera um texto final baseado na pesquisa.

Instruções: preencha os prompts e funções abaixo; crie o grafo e rode com um exemplo.


In [ ]:
class RWState(TypedDict):
    pergunta: str
    pesquisa: str
    texto: str

# TODO: crie o prompt do pesquisador
prompt_pesquisador = ChatPromptTemplate.from_messages([
    # ("system", "..."),
    # ("human", "Pergunta: {pergunta} ..."),
])

# TODO: crie o prompt do escritor
prompt_escritor = ChatPromptTemplate.from_messages([
    # ("system", "..."),
    # ("human", "Resumo: {pesquisa} ..."),
])

# TODO: implemente o nó pesquisador usando o LLM
def pesquisador(state: RWState):
    raise NotImplementedError("Implemente a chamada ao LLM para gerar pesquisa")

# TODO: implemente o nó escritor usando o LLM
def escritor(state: RWState):
    raise NotImplementedError("Implemente a chamada ao LLM para gerar texto")

# TODO: monte o grafo com os nós e arestas e compile


# TODO: invoque o grafo com uma pergunta do seu domínio e inspecione o resultado



## Exercício 2 — Debate com Árbitro
Crie dois agentes com perspectivas diferentes e um árbitro que sintetiza um resultado equilibrado.

Instruções: defina prompts de A e B e o árbitro; conecte os nós no grafo e execute.


In [ ]:
class DebateState(TypedDict):
    tema: str
    argumento_a: str
    argumento_b: str
    sintese: str

# TODO: defina prompt de A
prompt_a = ChatPromptTemplate.from_messages([
    # ("system", "..."),
    # ("human", "Tema: {tema} ..."),
])

# TODO: defina prompt de B
prompt_b = ChatPromptTemplate.from_messages([
    # ("system", "..."),
    # ("human", "Tema: {tema} ..."),
])

# TODO: defina prompt do árbitro
prompt_arbitro = ChatPromptTemplate.from_messages([
    # ("system", "..."),
    # ("human", "Argumento A: {argumento_a} ..."),
])

# TODO: implemente nós A, B e Árbitro usando o LLM
def agente_a(state: DebateState):
    raise NotImplementedError

def agente_b(state: DebateState):
    raise NotImplementedError

def arbitro(state: DebateState):
    raise NotImplementedError

# TODO: monte o grafo e execute um exemplo



## Exercício 3 — Hierárquico (Coordenador, Trabalhadores, Agregador)
Modele um coordenador que divide trabalho, dois agentes trabalhadores que produzem partes, e um agregador que compõe a saída final.

Instruções: crie prompts para cada papel, implemente os nós e conecte o grafo.


In [ ]:
class HierState(TypedDict):
    tarefa: str
    plano: str
    parte1: str
    parte2: str
    resultado: str

# TODO: crie os prompts de coordenador, trabalhador1, trabalhador2 e agregador
prompt_coord = ChatPromptTemplate.from_messages([
    # ...
])
prompt_worker1 = ChatPromptTemplate.from_messages([
    # ...
])
prompt_worker2 = ChatPromptTemplate.from_messages([
    # ...
])
prompt_agg = ChatPromptTemplate.from_messages([
    # ...
])

# TODO: implemente as funções dos nós chamando o LLM
def coord(state: HierState):
    raise NotImplementedError
def worker1(state: HierState):
    raise NotImplementedError
def worker2(state: HierState):
    raise NotImplementedError
def agg(state: HierState):
    raise NotImplementedError

# TODO: monte o grafo hierárquico e rode um exemplo


## Exercício 4 — Bônus (Produto + Prospecção com RAG)
Construa um fluxo simples que consulta o catálogo via RAG e gere um texto de prospecção.

Instruções: se desejar, reutilize `build_rag`; implemente os nós para diagnóstico, produto e e-mail.


In [ ]:
class SalesState(TypedDict):
    cliente: str
    contexto: str
    diagnostico: str
    produto_info: str
    texto_prospeccao: str

# TODO: defina os prompts de diagnóstico e de e-mail
prompt_diag = ChatPromptTemplate.from_messages([
    # ...
])
prompt_email = ChatPromptTemplate.from_messages([
    # ...
])

# TODO: implemente nós consultivo, especialista_produto (usando RAG opcional) e escritor
def consultivo(state: SalesState):
    raise NotImplementedError

def especialista_produto(state: SalesState):
    # Dica: se build_rag não estiver disponível, retorne mensagem informativa
    raise NotImplementedError

def escritor(state: SalesState):
    raise NotImplementedError

# TODO: monte o grafo e execute um exemplo com cliente e contexto



## Dicas
- Refine os prompts para o seu domínio e inclua restrições claras.
- Use memória de conversa (`RunnableWithMessageHistory`/buffers) se precisar de contexto entre execuções.
- Para avaliar qualidade, crie critérios objetivos e verifique se os agentes cumprem.
